# **Appendix Analysis for Fano Hypersurfaces**

Using `fano_main.py`, we obtain 5 runs of the dynamic search, and 5 runs of the ablation search.

This file has the following part: 

$\S1$ **Analysis**: We analyse the results of the dynamic and ablation searches

## $\S1$ Analysis

Load Packages

In [ ]:
#Storage
import numpy as np
import os
import pandas as pd
import json
#Figures
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

Load Data

In [ ]:
#Fixed Heuristic Search Data
fixed_reward_vs_steps = pd.read_csv(f"data/fixed_reward_vs_steps.csv", header=None).values.astype(int)
#Dynamic Heuristic Search Data
dynamic_settings = json.load(open("data/dynamic_settings.json"))
max_episodes = dynamic_settings["max_episodes"]
state_dim = dynamic_settings["state_dim"]
dynamic_reward_vs_steps_episodal = []
dynamic_differing_distances_episodal = []
for i in range(1, max_episodes + 1):
    dynamic_reward_vs_steps_episodal.append(pd.read_csv(f"data/dynamic_reward_vs_steps_episode_{i}.csv", header=None).values.astype(int))
    flat = pd.read_csv(f"data/dynamic_differing_distances_episode_{i}.csv", header=None).values
    distances = flat[:, -1].tolist()
    dynamic_differing_distances_episodal.append(distances)
#Ablation Search Data
ablation_settings = json.load(open("data/ablation_settings.json"))
max_episodes = ablation_settings["max_episodes"]
state_dim = ablation_settings["state_dim"]
ablation_reward_vs_steps_episodal = []
ablation_differing_distances_episodal = []
for i in range(1, max_episodes + 1):
    ablation_reward_vs_steps_episodal.append(pd.read_csv(f"data/ablation_reward_vs_steps_episode_{i}.csv", header=None).values.astype(int))
    flat = pd.read_csv(f"data/ablation_differing_distances_episode_{i}.csv", header=None).values
    distances = flat[:, -1].tolist()
    ablation_differing_distances_episodal.append(distances)

Dynamic Reward vs Steps

In [ ]:
# Font settings
plt.rcParams.update({
    "text.usetex": False,       
    "font.family": "serif",     
    "font.serif": ["DejaVu Serif"],  
    "mathtext.fontset": "cm",  
})

# Stack all runs into a single array of shape (max_episodes, 100_000)
all_rewards = np.array([dynamic_reward_vs_steps_episodal[i][:, 1] 
                        for i in range(max_episodes)])
steps = dynamic_reward_vs_steps_episodal[0][:, 0]  # Same for all runs

# Compute mean and std
mean_rewards = np.mean(all_rewards, axis=0)
std_rewards = np.std(all_rewards, axis=0, ddof=1)

# Instantiate plot
fig, ax = plt.subplots(figsize=(7, 6))

# Plot individual runs
for i in range(max_episodes):
    ax.plot(steps, all_rewards[i], color='blue', alpha=0.5, linewidth=1.5)

# Plot mean and shaded std
ax.plot(steps, mean_rewards, color='blue', linewidth=3, label='Mean')
ax.fill_between(steps, mean_rewards - std_rewards, mean_rewards + std_rewards,
                color='blue', alpha=0.15, linewidth=0, label=r'Mean $\pm$ Std')

# Format plot
ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}')) 
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax.grid(axis="y", alpha=0.3, linestyle='-')
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines['bottom'].set_color("grey")
ax.tick_params(axis='x', colors="grey")
ax.xaxis.label.set_color("grey")
ax.yaxis.label.set_color("grey")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.yaxis.set_tick_params(width=0)
plt.xticks(rotation=45)
plt.xticks(fontsize=18, color='grey')
plt.yticks(fontsize=18, color='grey')

# Labels and legend
ax.set_xlabel("Step Count", fontsize=20, color='grey', labelpad=15)
ax.set_ylabel("Terminal Points Found", fontsize=20, color='grey', labelpad=15)
plt.legend(frameon=False, fontsize=18, loc='lower right',
            title_fontsize=20)

plt.show()

os.makedirs("figures", exist_ok=True)
fig.savefig("figures/dynamic_rewards_vs_steps.png", dpi=300, bbox_inches="tight")

Dynamic Histograms

In [ ]:
# Global constants for alignment
global_min = int(min(min(d) for d in dynamic_differing_distances_episodal))
global_max = int(max(max(d) for d in dynamic_differing_distances_episodal))
global_bins = np.arange(global_min, global_max + 2) - 0.5
bin_centers_global = (global_bins[:-1] + global_bins[1:]) / 2

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "cm",
})

# --- CREATE COMBINED FIGURE (3 rows, 2 columns) ---
fig, axes = plt.subplots(2, 3, figsize=(30, 15)) # Adjusted size for 3x2
axes = axes.flatten()

all_counts_global_scale = [] 

# Plot Individual Runs (Indices 0-4)
for i in range(max_episodes):
    ax = axes[i]
    data = dynamic_differing_distances_episodal[i]
    
    local_max = int(max(data))
    local_bins = np.arange(global_min, local_max + 2) - 0.5
    local_centers = (local_bins[:-1] + local_bins[1:]) / 2

    counts, _, patches = ax.hist(data, bins=local_bins,
                                  color='blue', alpha=0.5, rwidth=0.85)
    
    # Capture counts on global scale for the mean calculation later
    g_counts, _ = np.histogram(data, bins=global_bins)
    all_counts_global_scale.append(g_counts)

    # Styling
    ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.grid(axis="y", alpha=0.3, linestyle='-')
    for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color("grey")
    
    ax.set_xticks(local_centers)
    ax.tick_params(axis='x', colors="grey", rotation=45, labelsize=14)
    ax.tick_params(axis='y', colors="grey", labelsize=14)
    
    ax.set_xlabel("Distance", fontsize=20, color='grey', labelpad=10)
    ax.set_ylabel("Frequency", fontsize=20, color='grey', labelpad=10)
    ax.set_title(f"Run {i+1}", fontsize=22, color='black', pad=15)
    ax.set_ylim(bottom=0)

    for patch, count in zip(patches, counts):
        if count > 0:
            ax.text(patch.get_x() + patch.get_width() / 2, patch.get_height(),
                    f'{int(count)}', ha='center', va='bottom', fontsize=12, color='grey')

# --- Plot Mean Summary (Index 5) ---
ax_mean = axes[5]
all_counts_global_scale = np.array(all_counts_global_scale)
mean_counts = np.mean(all_counts_global_scale, axis=0)
std_counts = np.std(all_counts_global_scale, axis=0, ddof=1)

ax_mean.bar(bin_centers_global, mean_counts, width=0.85, color='blue', alpha=0.7)
ax_mean.errorbar(bin_centers_global, mean_counts, yerr=std_counts,
                 fmt='none', color='black', capsize=4, linewidth=1.5, alpha=0.7)

# Styling Mean Plot
ax_mean.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax_mean.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax_mean.grid(axis="y", alpha=0.3, linestyle='-')
for s in ["top", "right", "left"]: ax_mean.spines[s].set_visible(False)
ax_mean.spines['bottom'].set_color("grey")

ax_mean.set_xticks(bin_centers_global)
ax_mean.tick_params(axis='x', colors="grey", rotation=45, labelsize=14)
ax_mean.tick_params(axis='y', colors="grey", labelsize=14)

ax_mean.set_xlabel("Distance", fontsize=20, color='grey', labelpad=10)
ax_mean.set_ylabel("Mean Frequency", fontsize=20, color='grey', labelpad=10)
ax_mean.set_title(f'Mean $\pm$ Std ', fontsize=22, color='black', fontweight='bold', pad=15)

# Formatting values above whiskers
def format_value(m_val):
    return int(m_val) if int(m_val) == m_val else round(m_val, 1)

for i, m_val in enumerate(mean_counts):
    if m_val > 0:
        display_y = m_val + std_counts[i]
        offset = (max(mean_counts + std_counts)) * 0.02
        ax_mean.text(bin_centers_global[i], display_y + offset, 
                     f'{format_value(m_val)}', ha='center', va='bottom', 
                     fontsize=12, color='black')

ax_mean.set_ylim(0, max(mean_counts + std_counts) * 1.1)

# Final Layout
plt.tight_layout(pad=4.0)
plt.show()

# Save combined figure
os.makedirs("figures", exist_ok=True)
fig.savefig("figures/dynamic_histogram.png", dpi=300, bbox_inches="tight")

Ablation Rewards vs Steps

In [ ]:
# Font settings
plt.rcParams.update({
    "text.usetex": False,       
    "font.family": "serif",     
    "font.serif": ["DejaVu Serif"],  
    "mathtext.fontset": "cm",  
})

# Instantiate plot
fig, ax = plt.subplots(figsize=(7, 6))

#Fixed
fixed_rewards = fixed_reward_vs_steps[:, 1]
fixed_steps = fixed_reward_vs_steps[:, 0]
ax.plot(fixed_steps, fixed_rewards, color='blue', alpha=0.5, linestyle='--', label='Fixed')


#Dynamic
dynamic_rewards_all = np.array([dynamic_reward_vs_steps_episodal[i][:, 1] 
                        for i in range(max_episodes)])
steps = dynamic_reward_vs_steps_episodal[0][:, 0]  
mean_rewards = np.mean(dynamic_rewards_all, axis=0)
std_rewards = np.std(dynamic_rewards_all, axis=0, ddof=1)
# Plot individual runs
for i in range(max_episodes):
    ax.plot(steps, dynamic_rewards_all[i], color='blue', alpha=0.5, linewidth=1.5)
# Plot mean and shaded std
ax.plot(steps, mean_rewards, color='blue', linewidth=3, label='Dyn. Mean')
ax.fill_between(steps, mean_rewards - std_rewards, mean_rewards + std_rewards,
                color='blue', alpha=0.15, linewidth=0, label=r'Dyn. Mean $\pm$ Std')

#Ablation
ablation_rewards_all = np.array([ablation_reward_vs_steps_episodal[i][:, 1] 
                        for i in range(max_episodes)])
ablation_steps = ablation_reward_vs_steps_episodal[0][:, 0]
ablation_mean_rewards = np.mean(ablation_rewards_all, axis=0)
ablation_std_rewards = np.std(ablation_rewards_all, axis=0, ddof=1)
# Plot individual runs
for i in range(max_episodes):
    ax.plot(ablation_steps, ablation_rewards_all[i], color='blue', linestyle='-.', alpha=0.5, linewidth=1.5)
# Plot mean and shaded std
ax.plot(ablation_steps, ablation_mean_rewards, color='blue', linewidth=3, linestyle='-.', label='Abl. Mean')
ax.fill_between(ablation_steps, ablation_mean_rewards - ablation_std_rewards, ablation_mean_rewards + ablation_std_rewards,
                color='blue', alpha=0.15, linewidth=0, hatch='xx', label=r'Abl. Mean $\pm$ Std')

# Format plot
ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}')) 
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax.grid(axis="y", alpha=0.3, linestyle='-')
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines['bottom'].set_color("grey")
ax.tick_params(axis='x', colors="grey")
ax.xaxis.label.set_color("grey")
ax.yaxis.label.set_color("grey")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.yaxis.set_tick_params(width=0)
plt.xticks(rotation=45)
plt.xticks(fontsize=18, color='grey')
plt.yticks(fontsize=18, color='grey')

# Labels and legend
ax.set_xlabel("Step Count", fontsize=20, color='grey', labelpad=15)
ax.set_ylabel("Terminal Points Found", fontsize=20, color='grey', labelpad=15)
plt.legend(frameon=False, fontsize=18, loc='lower right',
            title_fontsize=20)

plt.show()

os.makedirs("figures", exist_ok=True)
fig.savefig("figures/ablation_rewards_vs_steps.png", dpi=300, bbox_inches="tight")

Ablation Histogram

In [ ]:
# Global constants for alignment
global_min = int(min(min(d) for d in ablation_differing_distances_episodal))
global_max = int(max(max(d) for d in ablation_differing_distances_episodal))
global_bins = np.arange(global_min, global_max + 2) - 0.5
bin_centers_global = (global_bins[:-1] + global_bins[1:]) / 2

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "cm",
})

# --- CREATE COMBINED FIGURE (3 rows, 2 columns) ---
fig, axes = plt.subplots(2, 3, figsize=(30, 15)) # Adjusted size for 3x2
axes = axes.flatten()

all_counts_global_scale = [] 

# Plot Individual Runs (Indices 0-4)
for i in range(max_episodes):
    ax = axes[i]
    data = ablation_differing_distances_episodal[i]
    
    local_max = int(max(data))
    local_bins = np.arange(global_min, local_max + 2) - 0.5
    local_centers = (local_bins[:-1] + local_bins[1:]) / 2

    counts, _, patches = ax.hist(data, bins=local_bins,
                                  color='blue', alpha=0.5, rwidth=0.85)
    
    # Capture counts on global scale for the mean calculation later
    g_counts, _ = np.histogram(data, bins=global_bins)
    all_counts_global_scale.append(g_counts)

    # Styling
    ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.grid(axis="y", alpha=0.3, linestyle='-')
    for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color("grey")
    
    ax.set_xticks(local_centers)
    ax.tick_params(axis='x', colors="grey", rotation=45, labelsize=14)
    ax.tick_params(axis='y', colors="grey", labelsize=14)
    
    ax.set_xlabel("Distance", fontsize=20, color='grey', labelpad=10)
    ax.set_ylabel("Frequency", fontsize=20, color='grey', labelpad=10)
    ax.set_title(f"Run {i+1}", fontsize=22, color='black', pad=15)
    ax.set_ylim(bottom=0)

    for patch, count in zip(patches, counts):
        if count > 0:
            ax.text(patch.get_x() + patch.get_width() / 2, patch.get_height(),
                    f'{int(count)}', ha='center', va='bottom', fontsize=12, color='grey')

# --- Plot Mean Summary (Index 5) ---
ax_mean = axes[5]
all_counts_global_scale = np.array(all_counts_global_scale)
mean_counts = np.mean(all_counts_global_scale, axis=0)
std_counts = np.std(all_counts_global_scale, axis=0, ddof=1)

ax_mean.bar(bin_centers_global, mean_counts, width=0.85, color='blue', alpha=0.7)
ax_mean.errorbar(bin_centers_global, mean_counts, yerr=std_counts,
                 fmt='none', color='black', capsize=4, linewidth=1.5, alpha=0.7)

# Styling Mean Plot
ax_mean.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax_mean.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax_mean.grid(axis="y", alpha=0.3, linestyle='-')
for s in ["top", "right", "left"]: ax_mean.spines[s].set_visible(False)
ax_mean.spines['bottom'].set_color("grey")

ax_mean.set_xticks(bin_centers_global)
ax_mean.tick_params(axis='x', colors="grey", rotation=45, labelsize=14)
ax_mean.tick_params(axis='y', colors="grey", labelsize=14)

ax_mean.set_xlabel("Distance", fontsize=20, color='grey', labelpad=10)
ax_mean.set_ylabel("Mean Frequency", fontsize=20, color='grey', labelpad=10)
ax_mean.set_title(f'Mean $\pm$ Std ', fontsize=22, color='black', fontweight='bold', pad=15)

# Formatting values above whiskers
def format_value(m_val):
    return int(m_val) if int(m_val) == m_val else round(m_val, 1)

for i, m_val in enumerate(mean_counts):
    if m_val > 0:
        display_y = m_val + std_counts[i]
        offset = (max(mean_counts + std_counts)) * 0.02
        ax_mean.text(bin_centers_global[i], display_y + offset, 
                     f'{format_value(m_val)}', ha='center', va='bottom', 
                     fontsize=12, color='black')

ax_mean.set_ylim(0, max(mean_counts + std_counts) * 1.1)

# Final Layout
plt.tight_layout(pad=4.0)
plt.show()

# Save combined figure
os.makedirs("figures", exist_ok=True)
fig.savefig("figures/ablation_histogram.png", dpi=300, bbox_inches="tight")